# **Sel 1: Markdown**

# **1. Load Dataset Kaggle & Vocabulary**

# **Sel 2: Code**

In [9]:
import pandas as pd
import numpy as np
import re

# Sesuaikan nama file Kaggle-nya dengan yang kamu download
df_kaggle = pd.read_csv('tech_internship_applications.csv')
df_vocab = pd.read_csv('skill_vocabulary.csv')

print(f"Ukuran awal dataset Kaggle: {df_kaggle.shape}")

Ukuran awal dataset Kaggle: (300, 10)


# **Sel 3: Markdown**

**2. Handle Missing Values & Filter Khusus Data Intern**

# **Sel 4: Code**

In [10]:
# Drop baris jika kolom 'Skills' atau 'Internship_Domain' kosong
df_kaggle = df_kaggle.dropna(subset=['Skills', 'Internship_Domain'])

# FILTERING: Ambil hanya baris yang mengandung kata 'data', 'analytics', 'machine learning', atau 'ai'
# pada kolom Internship_Domain (ubah ke lowercase dulu agar pencarian tidak sensitif huruf besar/kecil)
kondisi_data = df_kaggle['Internship_Domain'].str.lower().str.contains('data|analytics|machine learning|ai', na=False)
df_kaggle = df_kaggle[kondisi_data]

# Reset index agar rapi setelah difilter
df_kaggle = df_kaggle.reset_index(drop=True)

print(f"Ukuran dataset KHUSUS DATA INTERN: {df_kaggle.shape}")
display(df_kaggle[['Internship_Domain', 'Skills']].head())

Ukuran dataset KHUSUS DATA INTERN: (125, 10)


,Internship_Domain,Skills
0,Data Science,"MongoDB, SQL, Docker, HTML, Pandas"
1,AI/ML,"Kubernetes, Python, C++, MongoDB"
2,Data Science,"SQL, Node.js, CSS, Docker, C++"
3,Data Science,"React, HTML, Pandas, CSS, Python"
4,AI/ML,"HTML, Kubernetes, JavaScript, C++"


# **Sel 5: Markdown**

## **3. Cleaning, Normalisasi & Penyelarasan Urutan Skills**

# **Sel 6: Code**

In [11]:
# Ambil list skill dari vocabulary untuk memastikan urutannya persis sama dengan data lokal
vocab_list = df_vocab['skill'].dropna().tolist()

def extract_and_sort_skills(raw_text):
    text = str(raw_text).lower()

    # Normalisasi penulisan
    text = text.replace('c++', 'cpp').replace('node.js', 'node')
    text = text.replace('react.js', 'react').replace('next.js', 'nextjs')

    matched = []
    # Looping pencarian menggunakan urutan dari vocab_list
    for v in vocab_list:
        pattern = r'\b' + re.escape(v) + r'\b'
        if re.search(pattern, text):
            matched.append(v)

    # Gabungkan dengan koma dan spasi
    return ", ".join(matched)

# Rename dan buat kolom baru
df_kaggle = df_kaggle.rename(columns={'Skills': 'raw_skills'})
df_kaggle['skills_cleaned'] = df_kaggle['raw_skills'].apply(extract_and_sort_skills)

display(df_kaggle[['raw_skills', 'skills_cleaned']].head())

,raw_skills,skills_cleaned
0,"MongoDB, SQL, Docker, HTML, Pandas","docker, html, mongodb, pandas, sql"
1,"Kubernetes, Python, C++, MongoDB","cpp, kubernetes, mongodb, python"
2,"SQL, Node.js, CSS, Docker, C++","cpp, css, docker, node, sql"
3,"React, HTML, Pandas, CSS, Python","css, html, pandas, python, react"
4,"HTML, Kubernetes, JavaScript, C++","cpp, html, javascript, kubernetes"


# **Sel 7: Markdown**

### **4. Feature Engineering (Role & Skills Count)**

# **Sel 8: Code**

In [12]:
# Hitung jumlah skill
df_kaggle['skills_count'] = df_kaggle['skills_cleaned'].apply(lambda x: len(x.split(', ')) if x != "" else 0)

# Karena kita sudah memfilter khusus data, mapping role akan difokuskan ke ranah data/AI
def map_role(domain):
    domain = str(domain).lower()
    if 'machine learning' in domain or 'ai' in domain:
        return 'ai/ml'
    elif 'analyst' in domain or 'analytics' in domain:
        return 'data analyst'
    elif 'engineer' in domain:
        return 'data engineer'
    else:
        return 'data scientist' # Default fallback untuk ranah data

df_kaggle['role'] = df_kaggle['Internship_Domain'].apply(map_role)

display(df_kaggle[['Internship_Domain', 'role', 'skills_count']].head())

,Internship_Domain,role,skills_count
0,Data Science,data scientist,5
1,AI/ML,ai/ml,4
2,Data Science,data scientist,5
3,Data Science,data scientist,5
4,AI/ML,ai/ml,4


# **Sel 9: Markdown**

### **5. Final Check & Save Dataset**

# **Sel 10: Code**

In [13]:
print("--- Pengecekan Terakhir ---")

# Susun ulang kolom agar sesuai format (sesuaikan nama kolom akhir dengan kebutuhan format lokal)
final_cols = [
    'Internship_Domain', 'role', 'raw_skills', 'skills_cleaned', 'skills_count'
]

# Ambil kolom yang relevan dan ada di dataframe
final_cols = [col for col in final_cols if col in df_kaggle.columns]
df_global_final = df_kaggle[final_cols]

df_global_final.info()

# Simpan hasil kerja ke CSV
nama_file_baru = 'global_data_intern_cleaned.csv'
df_global_final.to_csv(nama_file_baru, index=False)

print(f"\n✅ BERHASIL! File '{nama_file_baru}' khusus Data Intern sudah tersimpan dan urutan skills-nya sudah persis Magang-in lokal!")

--- Pengecekan Terakhir ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Internship_Domain  125 non-null    object
 1   role               125 non-null    object
 2   raw_skills         125 non-null    object
 3   skills_cleaned     125 non-null    object
 4   skills_count       125 non-null    int64 
dtypes: int64(1), object(4)
memory usage: 5.0+ KB

✅ BERHASIL! File 'global_data_intern_cleaned.csv' khusus Data Intern sudah tersimpan dan urutan skills-nya sudah persis Magang-in lokal!
